### DATA LOADING

In [11]:
import pandas as pd
import numpy as np

df4 = pd.read_csv('../../datasets/complete_data.csv')

print(df4.shape)
assert len(df4) == 215371, f"Jumlah baris tidak sesuai ekspektasi awal (215.371): dapat {len(df4)}"
print("Load berhasil, jumlah baris sesuai ekspektasi.")

(215371, 11)
Load berhasil, jumlah baris sesuai ekspektasi.


### TRIM WHITESPACE

In [12]:
string_cols = ['school_name', 'province_name', 'city_name', 'district_name']

for col in string_cols:
    df4[col] = df4[col].str.strip()

print("Trim selesai untuk kolom:", string_cols)

Trim selesai untuk kolom: ['school_name', 'province_name', 'city_name', 'district_name']


### DETECT AND FLAG INVALID COORDS

In [13]:
invalid_mask = (
    df4['lat'].isnull() | df4['long'].isnull() |
    ((df4['lat'] == 0.0) & (df4['long'] == 0.0)) |
    (df4['long'] > 141) | (df4['long'] < 95) |
    (df4['lat'] > 6) | (df4['lat'] < -11)
)

df4['has_valid_coord'] = ~invalid_mask
df4.loc[invalid_mask, ['lat', 'long']] = np.nan

n_invalid = invalid_mask.sum()
print(f"Baris dengan koordinat tidak valid: {n_invalid} ({n_invalid/len(df4)*100:.2f}%)")
assert n_invalid == 1866, f"Jumlah koordinat tidak valid berubah dari ekspektasi (1866): dapat {n_invalid}"

Baris dengan koordinat tidak valid: 1866 (0.87%)


### REMOVE DUPLICATES

In [14]:
before = len(df4)

df4 = df4.drop_duplicates(
    subset=['school_name', 'province_name', 'city_name', 'lat', 'long', 'stage'],
    keep='first'
).reset_index(drop=True)

after = len(df4)
removed = before - after

print(f"Baris sebelum: {before}, sesudah: {after}, dihapus: {removed}")
assert removed == 86, f"Jumlah duplikat terhapus berubah dari ekspektasi (86): dapat {removed}"
print("Deduplikasi sesuai ekspektasi.")

Baris sebelum: 215371, sesudah: 215285, dihapus: 86
Deduplikasi sesuai ekspektasi.


### KALIMANTAN SUBSET FILTER

In [15]:
kalimantan_provinces = ['KALIMANTAN BARAT', 'KALIMANTAN TENGAH', 'KALIMANTAN SELATAN', 'KALIMANTAN TIMUR', 'KALIMANTAN UTARA']

df4_kalimantan = df4[df4['province_name'].isin(kalimantan_provinces)].reset_index(drop=True)

print(f"Baris nasional (df4)          : {len(df4)}")
print(f"Baris Kalimantan (df4_kalimantan): {len(df4_kalimantan)}")
print(df4_kalimantan['province_name'].value_counts())

Baris nasional (df4)          : 215285
Baris Kalimantan (df4_kalimantan): 17548
province_name
KALIMANTAN BARAT      6247
KALIMANTAN SELATAN    3845
KALIMANTAN TENGAH     3809
KALIMANTAN TIMUR      2931
KALIMANTAN UTARA       716
Name: count, dtype: int64


### VALIDATION

In [16]:
# Validasi dtype
expected_dtypes = {
    'total_population': 'float64',
    'total_education_age_population': 'int64',
    'lat': 'float64',
    'long': 'float64',
    'has_valid_coord': 'bool'
}
for col, expected in expected_dtypes.items():
    actual = str(df4[col].dtype)
    assert actual == expected, f"{col}: dtype {actual}, harusnya {expected}"
print("✓ Dtype sesuai ekspektasi.")

# Validasi tidak ada duplikat tersisa
dupe_check = df4.duplicated(subset=['school_name','province_name','city_name','lat','long','stage']).sum()
assert dupe_check == 0, f"Masih ada duplikat: {dupe_check}"
print("✓ Tidak ada duplikat tersisa.")

# Validasi flag has_valid_coord konsisten dengan nilai NaN pada lat/long
mismatch = (
    (df4['has_valid_coord'] & (df4['lat'].isnull() | df4['long'].isnull())).sum()
    + (~df4['has_valid_coord'] & df4['lat'].notnull() & df4['long'].notnull()).sum()
)
assert mismatch == 0, f"Flag has_valid_coord tidak konsisten pada {mismatch} baris"
print("✓ Flag has_valid_coord konsisten.")

# Validasi df4_kalimantan adalah subset sah dari df4
assert df4_kalimantan['province_name'].isin(kalimantan_provinces).all()
assert len(df4_kalimantan) == df4['province_name'].isin(kalimantan_provinces).sum()
print("✓ Subset Kalimantan konsisten dengan df4 nasional.")

print(f"\nRingkasan akhir: {len(df4)} baris nasional, {len(df4_kalimantan)} baris Kalimantan.")

✓ Dtype sesuai ekspektasi.
✓ Tidak ada duplikat tersisa.
✓ Flag has_valid_coord konsisten.
✓ Subset Kalimantan konsisten dengan df4 nasional.

Ringkasan akhir: 215285 baris nasional, 17548 baris Kalimantan.


### EXPORT

In [17]:
import os

os.makedirs('../../datasets/processed', exist_ok=True)

df4.to_csv('../../datasets/processed/data4_schools_cleaned_national.csv', index=False)
df4_kalimantan.to_csv('../../datasets/processed/data4_schools_cleaned_kalimantan.csv', index=False)

print("Export selesai:")
print("- ../../datasets/processed/data4_schools_cleaned_national.csv")
print("- ../../datasets/processed/data4_schools_cleaned_kalimantan.csv")

Export selesai:
- ../../datasets/processed/data4_schools_cleaned_national.csv
- ../../datasets/processed/data4_schools_cleaned_kalimantan.csv
